In [1]:
# task2_churn_pipeline.py
# Submitted by : Saif Ullah
# Internship at Developers Hub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. LOAD DATA ====================
print("Step 1: Loading Data...")
# Download Telco Churn dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Quick look at data
print(f"Dataset shape: {df.shape}")
print(df.head(2))

# ==================== 2. PREPROCESSING ====================
print("\nStep 2: Preprocessing...")

# Handle missing values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Drop unnecessary columns
df.drop(['customerID'], axis=1, inplace=True)

# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})  # Convert to binary

# Identify column types
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

# ==================== 3. BUILD PIPELINE ====================
print("\nStep 3: Building Pipeline...")

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Preprocessing for numerical data
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# ==================== 4. TRAIN MODELS WITH GRIDSEARCH ====================
print("\nStep 4: Training Models...")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model 1: Logistic Regression Pipeline
print("Training Logistic Regression...")
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', LogisticRegression(random_state=42))])

# Model 2: Random Forest Pipeline with GridSearch
print("Training Random Forest with GridSearch...")
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', RandomForestClassifier(random_state=42))])

# Hyperparameter grid for Random Forest
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_split': [2, 5]
}

# GridSearch
grid_search = GridSearchCV(rf_pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Train simple Logistic Regression for comparison
lr_pipeline.fit(X_train, y_train)

# ==================== 5. EVALUATION ====================
print("\nStep 5: Evaluation...")

# Logistic Regression predictions
lr_pred = lr_pipeline.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)

# Best Random Forest from GridSearch
best_rf = grid_search.best_estimator_
rf_pred = best_rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print(f"Best RF Parameters: {grid_search.best_params_}")
print(f"\nLogistic Regression Accuracy: {lr_acc:.4f}")
print(f"Random Forest Accuracy: {rf_acc:.4f}")
print(f"\nRandom Forest Classification Report:")
print(classification_report(y_test, rf_pred))

# ==================== 6. EXPORT PIPELINE ====================
print("\nStep 6: Exporting Pipeline...")
joblib.dump(best_rf, 'churn_prediction_pipeline.pkl')
print("Pipeline saved as 'churn_prediction_pipeline.pkl'")

# Test loading the pipeline
loaded_pipeline = joblib.load('churn_prediction_pipeline.pkl')
test_pred = loaded_pipeline.predict(X_test[:5])
print(f"Sample predictions from loaded model: {test_pred}")

print("\n✅ Task 2 Complete!")

Step 1: Loading Data...
Dataset shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0          No          No              No  Month-to-month              Yes   
1          No          No              No        One year               No   

      PaymentMethod MonthlyCharges  TotalCharges Churn  
0  Electronic check          29.85         29.85    No  
1      Mailed check          56.95        1889.5    No  

[2 rows x 21 columns]

Step 2: Preprocessing...
Categorical columns: ['ge